# Notebook 3 — Advanced GANs: Pix2Pix, CycleGAN, and Evaluation Metrics

**Phase 3** deliverable. Learning objectives:
- Pix2Pix for paired image-to-image translation
- CycleGAN for unpaired style transfer
- FID and Inception Score evaluation

Complete the TODOs in `models/advanced/`, the Phase 3 losses in
`training/losses.py`, and `evaluation/metrics.py` first. Run
`pytest -m advanced -v` and `pytest -m evaluation -v` to check your progress.


In [1]:
import sys
from pathlib import Path

# Make `generative_art_studio` importable without `pip install -e .`
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
# Ensure the project root is used when the notebook is launched from the notebooks folder.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from generative_art_studio.utils import set_seed, plot_image_grid
from generative_art_studio.config import DEVICE

set_seed(42)
print(f"Using device: {DEVICE}")

Using device: cpu


## 1. Pix2Pix

Implement `UNetUp.forward`'s skip connection in `models/advanced/pix2pix.py`
and `pix2pix_generator_loss` in `training/losses.py`.


In [2]:
from generative_art_studio.models.advanced import UNetGenerator, PatchGANDiscriminator
from generative_art_studio.data import SyntheticImageDataset, get_dataloader

dataset = SyntheticImageDataset(num_samples=128, image_size=64)
dataloader = get_dataloader(dataset, batch_size=16)

unet = UNetGenerator(features=32).to(DEVICE)
patch_disc = PatchGANDiscriminator(in_channels=6).to(DEVICE)

images, _ = next(iter(dataloader))
images = images.to(DEVICE)
fake = unet(images)  # here input == "source" domain image; swap in a paired dataset for real use
print("Generated:", fake.shape)
plot_image_grid(fake.detach().cpu()[:8], nrow=4, title="Untrained Pix2Pix output")


Generated: torch.Size([16, 3, 64, 64])


<Figure size 400x400 with 1 Axes>

### TODO: Pix2Pix training loop

For a *paired* dataset (source_img, target_img), each step:
1. `fake = unet(source_img)`
2. Discriminator step: `patchgan_discriminator_loss(patch_disc(source_img, target_img), patch_disc(source_img, fake.detach()))`
3. Generator step: `pix2pix_generator_loss(patch_disc(source_img, fake), fake, target_img, lambda_l1=100.0)`


In [3]:
# TODO: your Pix2Pix training loop here (needs a paired dataset — see docs/PROJECT_BRIEF.md's WikiArt / Pix2Pix notes)
fake = unet(images)  # here input == "source" domain image; swap in a paired dataset for real use
#Discriminator loss
real_output = patch_disc(torch.cat((images, images), dim=1))  # Real
fake_output = patch_disc(torch.cat((images, fake.detach()), dim=1))  #Fake
disc_loss = -torch.mean(torch.log(real_output + 1e-8) + torch.log(1 - fake_output + 1e-8))
print("Discriminator loss:", disc_loss.item())
#Generator loss
gen_loss = -torch.mean(torch.log(fake_output + 1e-8))
print("Generator loss:", gen_loss.item())


Discriminator loss: nan
Generator loss: nan


## 2. CycleGAN

Implement `ResidualBlock.forward`'s skip connection in
`models/advanced/cyclegan.py` and `cycle_consistency_loss` /
`identity_loss` in `training/losses.py`.


In [4]:
from generative_art_studio.models.advanced import CycleGANGenerator

g_a2b = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
g_b2a = CycleGANGenerator(features=32, num_residual_blocks=4).to(DEVICE)
d_a = PatchGANDiscriminator(in_channels=3).to(DEVICE)  # unconditional: single image in
d_b = PatchGANDiscriminator(in_channels=3).to(DEVICE)

fake_b = g_a2b(images)
reconstructed_a = g_b2a(fake_b)
print("Cycle shapes:", images.shape, "->", fake_b.shape, "->", reconstructed_a.shape)


Cycle shapes: torch.Size([16, 3, 64, 64]) -> torch.Size([16, 3, 64, 64]) -> torch.Size([16, 3, 64, 64])


### TODO: CycleGAN training loop

For *unpaired* domain-A and domain-B images each step, combine:
- adversarial loss for both directions (`patchgan_discriminator_loss` / same-style generator adv term)
- `cycle_consistency_loss(real_a, g_b2a(g_a2b(real_a)))` (and the B->A->B direction)
- `identity_loss(real_b, g_a2b(real_b))` (and the mirror direction)


In [5]:
# TODO: your CycleGAN training loop here
#adversarial loss
real_output_a = d_a(images)  # Real A
fake_output_a = d_a(g_b2a(fake_b.detach()))  # Fake A
real_output_b = d_b(fake_b.detach())  # Real B
fake_output_b = d_b(g_a2b(images).detach())  # Fake B
disc_loss_a = -torch.mean(torch.log(real_output_a + 1e-8) + torch.log(1 - fake_output_a + 1e-8))
disc_loss_b = -torch.mean(torch.log(real_output_b + 1e-8) + torch.log(1 - fake_output_b + 1e-8))
print("Discriminator loss A:", disc_loss_a.item())
print("Discriminator loss B:", disc_loss_b.item())
gen_loss_a2b = -torch.mean(torch.log(fake_output_b + 1e-8))
gen_loss_b2a = -torch.mean(torch.log(fake_output_a + 1e-8))
print("Generator loss A->B:", gen_loss_a2b.item())
print("Generator loss B->A:", gen_loss_b2a.item())  

Discriminator loss A: nan
Discriminator loss B: nan
Generator loss A->B: nan
Generator loss B->A: nan


## 3. Evaluation: FID & Inception Score

In [6]:
from generative_art_studio.evaluation.metrics import compute_fid, compute_inception_score
import numpy as np

# Sanity check with synthetic features first (this is exactly what the graded tests do):
rng = np.random.default_rng(0)
real_features = rng.normal(size=(200, 32))
fake_features_good = real_features + rng.normal(scale=0.1, size=(200, 32))  # close to real
fake_features_bad = rng.normal(loc=5.0, size=(200, 32))  # far from real

print("FID (good generator):", compute_fid(real_features, fake_features_good))
print("FID (bad generator): ", compute_fid(real_features, fake_features_bad))


FID (good generator): 0.02841364788912705
FID (bad generator):  808.5358754507192


### TODO: real FID/IS on your trained models

```python
from generative_art_studio.evaluation.metrics import get_inception_feature_extractor

extractor = get_inception_feature_extractor().to(DEVICE)  # downloads pretrained weights
real_feats, _ = extractor(real_batch.to(DEVICE))
fake_feats, fake_probs = extractor(generated_batch.to(DEVICE))

fid = compute_fid(real_feats.cpu().numpy(), fake_feats.cpu().numpy())
is_mean, is_std = compute_inception_score(fake_probs.cpu().numpy())
```

Run this for at least two of your trained generators and compare.


In [7]:
# TODO: compute real FID/IS for your trained models here
from generative_art_studio.evaluation.metrics import get_inception_feature_extractor

extractor = get_inception_feature_extractor().to(DEVICE)  # downloads pretrained weights

# Use the images batch from earlier (real data)
real_batch = images
# Generate fake images using your trained generator (unet for Pix2Pix or g_a2b for CycleGAN)
generated_batch = unet(images)

real_feats, _ = extractor(real_batch.to(DEVICE))
fake_feats, fake_probs = extractor(generated_batch.to(DEVICE))

fid = compute_fid(real_feats.cpu().numpy(), fake_feats.cpu().numpy())
is_mean, is_std = compute_inception_score(fake_probs.cpu().numpy())

print(f"FID: {fid}")
print(f"Inception Score: {is_mean:.4f} ± {is_std:.4f}")

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to C:\Users\jbisw/.cache\torch\hub\checkpoints\inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:24<00:00, 4.51MB/s] 


FID: 27.72285030504092
Inception Score: 1.0030 ± 0.0091


c:\Users\jbisw\OneDrive - UNT System\Documents\Project5\generative-art-studio-tanvibbsr\src\generative_art_studio\evaluation\metrics.py:51: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  fid = np.sum((mu1 - mu2) ** 2) + np.trace(sigma1 + sigma2 - 2 * linalg.sqrtm(sigma1 @ sigma2)).real


## Reflection (for your Technical Report)

- Compare FID/IS across your models in one table.
- Where do FID/IS agree or disagree with your own visual judgment of
  quality? What might explain a disagreement?
